In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from src._01_code.features import build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

ModuleNotFoundError: No module named 'src'

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGETS = ["is_canceled", "adr"]

df = pd.read_csv("../data/clean/hotel_clean.csv")


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df["is_canceled"], random_state=RANDOM_STATE
)
print(f"train: {len(train_df):,} rows, cancel rate {train_df['is_canceled'].mean():.4f}")
print(f"test : {len(test_df):,} rows, cancel rate {test_df['is_canceled'].mean():.4f}")

# Features exclude BOTH targets: adr is the target for Module A, is_canceled for Module B
X_train = train_df.drop(columns=TARGETS)
X_test = test_df.drop(columns=TARGETS)

y_train, y_test = train_df["adr"], test_df["adr"]

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

train: 69,096 rows, cancel rate 0.2747
test : 17,275 rows, cancel rate 0.2747


In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean ADR learned : {baseline.constant_[0][0]:.2f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.2f}")

results = [{"model": "Mean baseline", "MAE": mae, "RMSE": rmse, "R2": r2}]

Mean ADR learned : 107.36
MAE: 39.38
RMSE: 51.07
R2: -0.00


In [ ]:
def make_pipe(model):
    return Pipeline([
        ("prep", build_preprocessor(num_cols, cat_cols)),
        ("model", model),
    ])

models = {"Linear Regression": make_pipe(LinearRegression())}